# Heck et al. 2025 – Jigsaw Puzzle Solver Baseline
### *"Solving Jigsaw Puzzles with Vision Transformers"*
Heck G., Lermé N., Le Hégarat-Mascle S. — Pattern Analysis and Applications (2025)

**Architecture:** CNN edge encoder + 6-layer Vision Transformer + Sinkhorn–Knopp permutation learning.

**Adaptation:** Applied to *curved* (jigsaw-tab) puzzle pieces instead of the paper's plain square pieces.

---
**Workflow:**
1. Pre-generate puzzle pieces locally: `python Generator/generate_dataset.py generate ./Dataset ./Dataset_Split --style curved`
2. Upload `train_set_curved_baseline/` to Google Drive.
3. Set `DATASET_PATH` in the **Config** cell, then run top-to-bottom.


In [ ]:
# ── Install (only missing packages) ─────────────────────────────────────────
!pip install -q scikit-image scipy tqdm Pillow matplotlib

# ── Imports ─────────────────────────────────────────────────────────────────
import os, math, random, time, json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from scipy.optimize import linear_sum_assignment
from scipy.stats import kendalltau as scipy_kendalltau
from skimage.metrics import structural_similarity as skimage_ssim

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# ── Google Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
@dataclass
class Config:
    # ── Paths (edit before running) ──────────────────────────────────────────
    DATASET_PATH:    str = "/content/drive/MyDrive/train_set_curved_baseline"
    CHECKPOINT_DIR:  str = "/content/drive/MyDrive/heck2025_checkpoints"

    # ── Puzzle structure ──────────────────────────────────────────────────────
    N_PIECES:  int = 20   # must match --num-pieces used during generation
    GRID_ROWS: int = 4    # int(math.sqrt(N_PIECES))
    GRID_COLS: int = 5    # math.ceil(N_PIECES / GRID_ROWS)

    # ── Piece preprocessing ───────────────────────────────────────────────────
    PIECE_SIZE: int = 128   # resize every piece to PIECE_SIZE × PIECE_SIZE (square)
    STRIP_W:    int = 8     # edge-strip width in pixels fed to the CNN

    # ── CNN edge encoder ─────────────────────────────────────────────────────
    EMBED_DIM: int = 320    # per-edge embedding dimension (paper: 320)

    # ── Transformer ───────────────────────────────────────────────────────────
    D:        int = 1024    # hidden size (paper: 1024)
    N_HEADS:  int = 16      # attention heads (paper: 16)
    N_LAYERS: int = 6       # encoder layers (paper: 6)

    # ── Sinkhorn–Knopp ────────────────────────────────────────────────────────
    SINKHORN_K:   int   = 20     # normalisation iterations (paper: 20)
    TAU:          float = 0.1    # temperature (paper: 0.1)

    # ── Phase 1 – encoder triplet training ───────────────────────────────────
    MARGIN_ALPHA:         float = 0.5    # triplet margin (paper: 0.5)
    LR_PHASE1:            float = 1e-4
    BATCH_SIZE_TRIPLETS:  int   = 128
    EPOCHS_PHASE1:        int   = 20

    # ── Phase 2 – global model training ──────────────────────────────────────
    LR_PHASE2:            float = 1e-4
    BATCH_SIZE_PUZZLES:   int   = 8      # reduce to 4 if OOM
    EPOCHS_PHASE2:        int   = 30

    # ── Data split ────────────────────────────────────────────────────────────
    TRAIN_RATIO:  float = 0.8
    NUM_WORKERS:  int   = 2      # set to 0 when data lives on Drive
    SEED:         int   = 42


cfg = Config()

# ── Validate grid dimensions ─────────────────────────────────────────────────
_rows = max(1, int(math.sqrt(cfg.N_PIECES)))
_cols = math.ceil(cfg.N_PIECES / _rows)
assert cfg.GRID_ROWS == _rows and cfg.GRID_COLS == _cols, (
    f"GRID_ROWS/GRID_COLS mismatch: expected {_rows}x{_cols}, "
    f"got {cfg.GRID_ROWS}x{cfg.GRID_COLS}"
)

Path(cfg.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

# ── Global seeds ──────────────────────────────────────────────────────────────
torch.manual_seed(cfg.SEED)
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

print("Config:")
for k, v in cfg.__dict__.items():
    print(f"  {k:<25} {v}")


In [ ]:
# ── Piece loading helpers ─────────────────────────────────────────────────────

def rgba_to_rgb(img: Image.Image) -> Image.Image:
    """Composite an RGBA image onto a white background → RGB."""
    if img.mode == "RGB":
        return img
    if img.mode != "RGBA":
        return img.convert("RGB")
    bg = Image.new("RGB", img.size, (255, 255, 255))
    bg.paste(img, mask=img.split()[3])
    return bg


def load_piece(path: Path, size: int) -> torch.Tensor:
    """Load one puzzle piece PNG → float32 [3, size, size] tensor in [0, 1]."""
    img = rgba_to_rgb(Image.open(path))
    img = img.resize((size, size), Image.LANCZOS)
    return T.ToTensor()(img)   # [3, size, size]


# ── Edge-strip extraction ─────────────────────────────────────────────────────

def get_edge_strip(piece: torch.Tensor, direction: str, strip_w: int) -> torch.Tensor:
    """
    Extract an edge strip from piece [3, H, W], applying a geometric transform
    so the edge of interest always ends up on the RIGHT side of the output strip.

    Transforms applied (ensures matching edges yield structurally similar strips):
        right  – identity                      → [3, H,    strip_w]
        left   – flip horizontal               → [3, H,    strip_w]
        bottom – rot90 CCW  (bottom → right)   → [3, W,    strip_w]
        top    – flip_v + rot90 CCW            → [3, W,    strip_w]

    For square pieces (H == W) all strips share the same shape [3, H, strip_w].
    """
    if direction == "right":
        return piece[..., -strip_w:]
    elif direction == "left":
        return piece.flip(-1)[..., -strip_w:]
    elif direction == "bottom":
        # rot90 CCW: [3,H,W] → [3,W,H]; right strip_w cols = bottom strip_w rows
        return piece.rot90(1, dims=[-2, -1])[..., -strip_w:]
    elif direction == "top":
        # flip_v then rot90 CCW: top rows → right columns (reversed)
        return piece.flip(-2).rot90(1, dims=[-2, -1])[..., -strip_w:]
    else:
        raise ValueError(f"Unknown direction: {direction!r}")


In [ ]:
# ── PuzzleDataset ─────────────────────────────────────────────────────────────

class PuzzleDataset(Dataset):
    """
    Loads pre-generated curved jigsaw puzzle piece sets.

    Expected layout:
        <DATASET_PATH>/
            <puzzle_name>/
                pieces/
                    piece_000.png  ...  piece_NNN.png

    __getitem__ randomly shuffles the N pieces and returns:
        shuffled_pieces  : [N, 3, S, S]  – shuffled input
        P_target         : [N, N]         – ground-truth permutation matrix
                           P_target[i, j] = 1 iff shuffled[j] belongs at position i
        ordered_pieces   : [N, 3, S, S]  – ground-truth ordered pieces
    """

    def __init__(self, puzzle_dirs: List[Path], cfg: Config, augment: bool = False):
        self.puzzle_dirs = puzzle_dirs
        self.cfg = cfg
        self.color_jitter = (
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)
            if augment else None
        )

    def load_ordered_pieces(self, pieces_dir: Path) -> torch.Tensor:
        """Load all N pieces in ground-truth order → [N, 3, S, S]."""
        pieces = []
        for i in range(self.cfg.N_PIECES):
            t = load_piece(pieces_dir / f"piece_{i:03d}.png", self.cfg.PIECE_SIZE)
            if self.color_jitter is not None and random.random() > 0.5:
                t = self.color_jitter(t)
            pieces.append(t)
        return torch.stack(pieces)  # [N, 3, S, S]

    def __len__(self) -> int:
        return len(self.puzzle_dirs)

    def __getitem__(self, idx: int):
        pieces_dir = self.puzzle_dirs[idx] / "pieces"
        ordered    = self.load_ordered_pieces(pieces_dir)  # [N, 3, S, S]
        N          = self.cfg.N_PIECES

        perm     = torch.randperm(N)       # perm[j] = original position of shuffled[j]
        shuffled = ordered[perm]            # [N, 3, S, S]

        # P_target[perm[j], j] = 1 for all j  (piece j came from position perm[j])
        P_target = torch.zeros(N, N)
        P_target[perm, torch.arange(N)] = 1.0

        return shuffled, P_target, ordered


# ── TripletDataset (Phase 1) ──────────────────────────────────────────────────

class TripletDataset(Dataset):
    """
    Generates (anchor, positive, negative) edge-strip triplets for the
    encoder pre-training phase.

    Right-adjacency  (A left, B right):
        anchor   = A's right edge strip
        positive = B's left  edge strip  (same boundary, other side)
        negative = random piece's left edge strip

    Bottom-adjacency (A top, B bottom):
        anchor   = A's bottom edge strip
        positive = B's top   edge strip
        negative = random piece's top edge strip
    """

    def __init__(self, base_ds: PuzzleDataset, cfg: Config):
        self.base = base_ds
        self.cfg  = cfg
        rows, cols = cfg.GRID_ROWS, cfg.GRID_COLS

        # (puzzle_idx, piece_a, piece_b, anchor_dir, pos_dir)
        self.specs: List[Tuple[int, int, int, str, str]] = []
        for pi in range(len(base_ds)):
            for r in range(rows):
                for c in range(cols):
                    a = r * cols + c
                    if c + 1 < cols:
                        self.specs.append((pi, a, r * cols + (c + 1), "right", "left"))
                    if r + 1 < rows:
                        self.specs.append((pi, a, (r + 1) * cols + c, "bottom", "top"))

    def __len__(self) -> int:
        return len(self.specs)

    def __getitem__(self, idx: int):
        pi, a_idx, b_idx, anchor_dir, pos_dir = self.specs[idx]
        pieces = self.base.load_ordered_pieces(
            self.base.puzzle_dirs[pi] / "pieces"
        )  # [N, 3, S, S]
        N  = self.cfg.N_PIECES
        sw = self.cfg.STRIP_W

        anchor   = get_edge_strip(pieces[a_idx], anchor_dir, sw)
        positive = get_edge_strip(pieces[b_idx], pos_dir,    sw)

        neg_idx = b_idx
        while neg_idx == b_idx:
            neg_idx = random.randint(0, N - 1)
        negative = get_edge_strip(pieces[neg_idx], pos_dir, sw)

        return anchor, positive, negative


# ── Dataset factory ────────────────────────────────────────────────────────────

def make_datasets(cfg: Config) -> Tuple[PuzzleDataset, PuzzleDataset]:
    """Scan DATASET_PATH and return (train_ds, test_ds)."""
    base = Path(cfg.DATASET_PATH)
    if not base.exists():
        raise FileNotFoundError(f"Dataset path not found: {base}")

    puzzle_dirs = sorted([
        d for d in base.iterdir()
        if d.is_dir()
        and (d / "pieces").is_dir()
        and len(list((d / "pieces").glob("piece_*.png"))) == cfg.N_PIECES
    ])

    if not puzzle_dirs:
        raise RuntimeError(
            f"No complete puzzles found in {base}.\n"
            f"Expected: <name>/pieces/piece_000.png … piece_{cfg.N_PIECES - 1:03d}.png"
        )

    rng = random.Random(cfg.SEED)
    rng.shuffle(puzzle_dirs)
    n_train   = int(len(puzzle_dirs) * cfg.TRAIN_RATIO)
    train_ds  = PuzzleDataset(puzzle_dirs[:n_train], cfg, augment=True)
    test_ds   = PuzzleDataset(puzzle_dirs[n_train:], cfg, augment=False)

    print(f"Found {len(puzzle_dirs)} puzzles → train: {n_train}, test: {len(test_ds)}")
    return train_ds, test_ds


# ── Quick sanity check (run once to verify paths) ─────────────────────────────
# train_ds, test_ds = make_datasets(cfg)
# shuffled, P_target, ordered = train_ds[0]
# print(f"shuffled : {shuffled.shape}")
# print(f"P_target : {P_target.shape}  row-sums={P_target.sum(1).tolist()[:4]}")
# print(f"ordered  : {ordered.shape}")


In [ ]:
# ── Sinkhorn–Knopp operator (Eq. 3 from paper) ────────────────────────────────

def sinkhorn_knopp(
    x: torch.Tensor,
    k: int   = 20,
    tau: float = 0.1,
) -> torch.Tensor:
    """
    Differentiable approximate Sinkhorn operator.

    Args:
        x   : [*, N, N] score matrix (can be batched)
        k   : number of alternating row/col normalisation steps
        tau : temperature  (lower → sharper / closer to a hard permutation)

    Returns:
        [*, N, N] doubly-stochastic matrix
    """
    s = torch.exp(x / tau)
    for _ in range(k):
        s = s / (s.sum(dim=-1, keepdim=True) + 1e-8)   # row normalise
        s = s / (s.sum(dim=-2, keepdim=True) + 1e-8)   # col normalise
    return s


# ── Hungarian decoding (inference only) ──────────────────────────────────────

@torch.no_grad()
def hungarian_decode(
    logits: torch.Tensor,
    tau:    float = 0.01,
    k:      int   = 100,
) -> np.ndarray:
    """
    Decode [B, N, N] logits to hard permutation assignments via the
    Hungarian algorithm.

    Returns
    -------
    assignments : int array [B, N]
        assignments[b, i] = j  means input piece j is placed at position i
                                in puzzle b.
    """
    B, N, _ = logits.shape
    # Use a sharper Sinkhorn to make the matrix near-permutation
    ds = sinkhorn_knopp(logits, k=k, tau=tau)
    ds_np       = ds.detach().cpu().numpy()
    assignments = np.empty((B, N), dtype=np.int64)
    for b in range(B):
        _, col_ind = linear_sum_assignment(-ds_np[b])   # maximise → negate cost
        assignments[b] = col_ind
    return assignments


In [ ]:
# ── CNN edge encoder (Section 3.2) ────────────────────────────────────────────

class EdgeCNN(nn.Module):
    """
    Encodes a single edge strip [B, 3, H, strip_w] → [B, embed_dim].

    Architecture (paper):
        Conv(3×3)+ReLU → MaxPool(2×2)   [×2]
        Conv(3×3)+ReLU
        AdaptiveAvgPool(4×4)
        FC → embed_dim

    AdaptiveAvgPool makes this resolution-independent (handles any strip size).
    """

    def __init__(self, embed_dim: int = 320):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(3,   64,  3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc   = nn.Linear(256 * 4 * 4, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.convs(x)     # [B, 256, H', W']
        x = self.pool(x)      # [B, 256, 4, 4]
        return self.fc(x.flatten(1))  # [B, embed_dim]


class PieceEncoder(nn.Module):
    """
    Encodes a puzzle piece [3, S, S] into a d-dimensional vector by:
        1. Extracting edge strips for all 4 directions (right/left/bottom/top).
        2. Encoding each strip with the shared EdgeCNN  → 4 × embed_dim vectors.
        3. Concatenating and projecting through an FC layer → d-dim vector.

    The single shared CNN + geometric transforms mirror the paper's approach of
    using axial/rotational symmetry to handle all four edge directions.
    """

    def __init__(self, embed_dim: int = 320, d: int = 1024, strip_w: int = 8):
        super().__init__()
        self.strip_w = strip_w
        self.cnn  = EdgeCNN(embed_dim)
        self.proj = nn.Linear(4 * embed_dim, d)

    @staticmethod
    def _extract_strips(pieces: torch.Tensor, strip_w: int) -> Tuple[
            torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return (right, left, bottom, top) strips for [B, 3, S, S] batch."""
        r = pieces[..., -strip_w:]
        l = pieces.flip(-1)[..., -strip_w:]
        b = pieces.rot90(1, dims=[-2, -1])[..., -strip_w:]
        t = pieces.flip(-2).rot90(1, dims=[-2, -1])[..., -strip_w:]
        return r, l, b, t

    def encode_strips(self, pieces: torch.Tensor) -> torch.Tensor:
        """pieces: [B, 3, S, S]  →  [B, d]"""
        r, l, b, t = self._extract_strips(pieces, self.strip_w)
        z = torch.cat([self.cnn(r), self.cnn(l), self.cnn(b), self.cnn(t)], dim=-1)
        return self.proj(z)  # [B, d]

    def forward(self, pieces: torch.Tensor) -> torch.Tensor:
        """pieces: [B, N, 3, S, S]  →  [B, N, d]"""
        B, N, C, S, _ = pieces.shape
        z = self.encode_strips(pieces.reshape(B * N, C, S, S))  # [B*N, d]
        return z.reshape(B, N, -1)                               # [B, N, d]


In [ ]:
# ── Transformer permutation network (Section 3.1) ─────────────────────────────

class PuzzleTransformer(nn.Module):
    """
    6-layer Transformer encoder with NO positional encoding (preserving the
    permutation invariance of the input), followed by an FC head that maps
    each piece token to N position logits.

    Input  : [B, N, d]
    Output : [B, N, N]   (logits[b, i, j] = score for piece i being at position j)
    """

    def __init__(
        self,
        d:        int = 1024,
        n_heads:  int = 16,
        n_layers: int = 6,
        n_pieces: int = 20,
        dropout:  float = 0.1,
    ):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=4 * d,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,          # pre-LN (ViT style)
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=n_layers, enable_nested_tensor=False
        )
        self.norm = nn.LayerNorm(d)
        self.fc   = nn.Linear(d, n_pieces)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, N, d]  (no positional encoding)  →  [B, N, N]"""
        z = self.norm(self.transformer(x))
        return self.fc(z)   # [B, N, N]


# ── End-to-end solver ─────────────────────────────────────────────────────────

class PuzzleSolver(nn.Module):
    """
    Full pipeline: PieceEncoder → PuzzleTransformer → Sinkhorn–Knopp.

    During training  : forward() returns a differentiable doubly-stochastic
                       matrix for the reconstruction loss.
    During inference : pass logits to hungarian_decode() for exact assignment.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.encoder     = PieceEncoder(cfg.EMBED_DIM, cfg.D, cfg.STRIP_W)
        self.transformer = PuzzleTransformer(
            cfg.D, cfg.N_HEADS, cfg.N_LAYERS, cfg.N_PIECES
        )
        self.cfg = cfg

    def forward(
        self,
        pieces: torch.Tensor,
        tau: Optional[float] = None,
        k:   Optional[int]   = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        pieces : [B, N, 3, S, S]
        returns: (logits [B, N, N],  ds_matrix [B, N, N])
        """
        tau = tau if tau is not None else self.cfg.TAU
        k   = k   if k   is not None else self.cfg.SINKHORN_K

        z      = self.encoder(pieces)       # [B, N, d]
        logits = self.transformer(z)        # [B, N, N]
        ds     = sinkhorn_knopp(logits, k, tau)
        return logits, ds


# ── Loss functions ─────────────────────────────────────────────────────────────

def cosine_dissimilarity(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Element-wise  1 − cos_sim(a, b).  Shape: same as inputs minus last dim."""
    return 1.0 - F.cosine_similarity(a, b, dim=-1)


def triplet_loss(
    za: torch.Tensor,
    zp: torch.Tensor,
    zn: torch.Tensor,
    margin: float = 0.5,
) -> torch.Tensor:
    """Triplet loss with cosine dissimilarity (Eq. 4 from paper)."""
    return F.relu(cosine_dissimilarity(za, zp) - cosine_dissimilarity(za, zn) + margin).mean()


def reconstruction_loss(
    ds:       torch.Tensor,   # [B, N, N]  predicted doubly-stochastic
    P_target: torch.Tensor,   # [B, N, N]  ground-truth permutation
    pieces:   torch.Tensor,   # [B, N, 3, S, S]
) -> torch.Tensor:
    """
    Image reconstruction L2 loss (Section 3.1):
        L = || P_bar(M) − P_pred(M) ||²

    Each position's 'image' is a linear combination of pieces weighted by the
    row of the permutation matrix, giving a differentiable proxy image.
    """
    B, N, C, S, _ = pieces.shape
    flat     = pieces.detach().reshape(B, N, -1)   # [B, N, C*S²] – no grad through pixels
    pred_img = torch.bmm(ds,       flat)             # [B, N, C*S²]
    gt_img   = torch.bmm(P_target, flat)             # [B, N, C*S²]
    return F.mse_loss(pred_img, gt_img)


In [ ]:
# ── Phase 1: encoder pre-training (triplet loss) ──────────────────────────────

def train_encoder(cfg: Config, train_ds: PuzzleDataset):
    """
    Train the CNN edge encoder alone with triplet loss.
    Saves checkpoint to cfg.CHECKPOINT_DIR/encoder_phase1.pt.
    """
    triplet_ds = TripletDataset(train_ds, cfg)
    loader = DataLoader(
        triplet_ds,
        batch_size=cfg.BATCH_SIZE_TRIPLETS,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    encoder = PieceEncoder(cfg.EMBED_DIM, cfg.D, cfg.STRIP_W).to(device)
    optim   = torch.optim.Adam(encoder.parameters(), lr=cfg.LR_PHASE1)
    scaler  = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    ckpt    = Path(cfg.CHECKPOINT_DIR) / "encoder_phase1.pt"
    history = {"loss": [], "top1_acc": []}

    print(f"\n{'='*60}")
    print(f"Phase 1 – encoder pre-training")
    print(f"  Triplets : {len(triplet_ds):,}")
    print(f"  Epochs   : {cfg.EPOCHS_PHASE1}")
    print(f"  Batch    : {cfg.BATCH_SIZE_TRIPLETS}")
    print(f"{'='*60}")

    for epoch in range(1, cfg.EPOCHS_PHASE1 + 1):
        encoder.train()
        run_loss, correct, total = 0.0, 0, 0
        t0 = time.time()

        for anchor, positive, negative in tqdm(
                loader, desc=f"Epoch {epoch}/{cfg.EPOCHS_PHASE1}", leave=False):
            anchor, positive, negative = (
                anchor.to(device), positive.to(device), negative.to(device)
            )
            optim.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                za = encoder.cnn(anchor)
                zp = encoder.cnn(positive)
                zn = encoder.cnn(negative)
                loss = triplet_loss(za, zp, zn, margin=cfg.MARGIN_ALPHA)

            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()

            run_loss += loss.item()
            with torch.no_grad():
                d_pos = cosine_dissimilarity(za, zp)
                d_neg = cosine_dissimilarity(za, zn)
                correct += (d_pos < d_neg).sum().item()
                total   += anchor.size(0)

        avg_loss = run_loss / len(loader)
        top1_acc = correct / total * 100
        history["loss"].append(avg_loss)
        history["top1_acc"].append(top1_acc)
        print(f"  Epoch {epoch:2d}/{cfg.EPOCHS_PHASE1} │ "
              f"loss={avg_loss:.4f}  top-1={top1_acc:.1f}%  ({time.time()-t0:.0f}s)")

    torch.save({"encoder_state": encoder.state_dict(),
                "history": history, "cfg": cfg.__dict__}, ckpt)
    print(f"\nCheckpoint saved → {ckpt}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["loss"]);    ax1.set(title="Triplet Loss",  xlabel="Epoch")
    ax2.plot(history["top1_acc"]); ax2.set(title="Top-1 Accuracy (%)", xlabel="Epoch")
    plt.tight_layout(); plt.show()
    return encoder


# ── Run Phase 1 ──────────────────────────────────────────────────────────────
# Uncomment to run (or load checkpoint below if already trained):
#
# train_ds, test_ds = make_datasets(cfg)
# encoder = train_encoder(cfg, train_ds)


In [ ]:
# ── Phase 2: global model training (image reconstruction loss) ────────────────

def train_global(
    cfg: Config,
    train_ds: PuzzleDataset,
    encoder_ckpt: Optional[str] = None,
) -> "PuzzleSolver":
    """
    Train the complete PuzzleSolver (encoder + ViT) jointly with image
    reconstruction loss.  Optionally warm-starts the encoder from Phase 1.
    Saves the best checkpoint to cfg.CHECKPOINT_DIR/global_model.pt.
    """
    loader = DataLoader(
        train_ds,
        batch_size=cfg.BATCH_SIZE_PUZZLES,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model  = PuzzleSolver(cfg).to(device)
    optim  = torch.optim.Adam(model.parameters(), lr=cfg.LR_PHASE2)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    ckpt   = Path(cfg.CHECKPOINT_DIR) / "global_model.pt"

    # ── Load Phase 1 encoder weights ─────────────────────────────────────────
    if encoder_ckpt and Path(encoder_ckpt).exists():
        saved = torch.load(encoder_ckpt, map_location=device)
        missing, unexpected = model.encoder.load_state_dict(
            saved["encoder_state"], strict=False
        )
        print(f"Loaded Phase 1 encoder  →  {encoder_ckpt}")
        if missing:    print(f"  Missing   : {missing}")
        if unexpected: print(f"  Unexpected: {unexpected}")
    else:
        print("No Phase 1 checkpoint  →  training from random init.")

    best_loss = float("inf")
    history   = {"loss": []}

    print(f"\n{'='*60}")
    print(f"Phase 2 – global model training")
    print(f"  Puzzles : {len(train_ds):,}")
    print(f"  Epochs  : {cfg.EPOCHS_PHASE2}")
    print(f"  Batch   : {cfg.BATCH_SIZE_PUZZLES}")
    print(f"{'='*60}")

    for epoch in range(1, cfg.EPOCHS_PHASE2 + 1):
        model.train()
        run_loss = 0.0
        t0 = time.time()

        for shuffled, P_target, ordered in tqdm(
                loader, desc=f"Epoch {epoch}/{cfg.EPOCHS_PHASE2}", leave=False):
            shuffled = shuffled.to(device)   # [B, N, 3, S, S]
            P_target = P_target.to(device)   # [B, N, N]

            optim.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                _, ds = model(shuffled)
                loss  = reconstruction_loss(ds, P_target, shuffled)

            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            run_loss += loss.item()

        avg_loss = run_loss / len(loader)
        history["loss"].append(avg_loss)
        print(f"  Epoch {epoch:2d}/{cfg.EPOCHS_PHASE2} │ "
              f"loss={avg_loss:.6f}  ({time.time()-t0:.0f}s)")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({"model_state": model.state_dict(),
                        "history": history, "epoch": epoch,
                        "cfg": cfg.__dict__}, ckpt)

    print(f"\nBest checkpoint saved → {ckpt}  (loss={best_loss:.6f})")
    plt.figure(figsize=(8, 4))
    plt.plot(history["loss"])
    plt.title("Phase 2 – Reconstruction Loss"); plt.xlabel("Epoch"); plt.ylabel("MSE")
    plt.tight_layout(); plt.show()
    return model


# ── Run Phase 2 ────────────────────────────────────────────────────────────────
# Uncomment to run (adjust encoder_ckpt path if Phase 1 was already run):
#
# train_ds, test_ds = make_datasets(cfg)    # skip if already loaded above
# encoder_ckpt = str(Path(cfg.CHECKPOINT_DIR) / "encoder_phase1.pt")
# model = train_global(cfg, train_ds, encoder_ckpt=encoder_ckpt)


In [ ]:
# ── Evaluation metrics ────────────────────────────────────────────────────────

def compute_acc(pred_asst: np.ndarray, gt_asst: np.ndarray) -> float:
    """ACC: fraction of positions with the exact correct piece."""
    return float(np.mean(pred_asst == gt_asst))


def compute_neighbor_perf(
    orig_at_pos: np.ndarray, rows: int, cols: int
) -> float:
    """
    NP: fraction of adjacent-position pairs where the assigned pieces are
    also grid-adjacent in ground truth.

    orig_at_pos[i] = original grid index of the piece placed at position i.
    """
    correct, total = 0, 0
    for i in range(rows * cols):
        r_i, c_i = divmod(i, cols)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            r_j, c_j = r_i + dr, c_i + dc
            if 0 <= r_j < rows and 0 <= c_j < cols:
                j = r_j * cols + c_j
                ri_o, ci_o = divmod(int(orig_at_pos[i]), cols)
                rj_o, cj_o = divmod(int(orig_at_pos[j]), cols)
                if abs(ri_o - rj_o) + abs(ci_o - cj_o) == 1:
                    correct += 1
                total += 1
    return correct / total if total > 0 else 0.0


def compute_kendall_tau(orig_at_pos: np.ndarray) -> float:
    """Kendall's τ between predicted and identity ordering."""
    tau, _ = scipy_kendalltau(orig_at_pos, np.arange(len(orig_at_pos)))
    return float(tau)


def pieces_to_grid(
    pieces_np: np.ndarray, rows: int, cols: int
) -> np.ndarray:
    """Stack [N, S, S, 3] pieces into a [rows*S, cols*S, 3] grid image."""
    N, S, _, C = pieces_np.shape
    grid = np.zeros((rows * S, cols * S, C), dtype=pieces_np.dtype)
    for i in range(min(N, rows * cols)):
        r, c = divmod(i, cols)
        grid[r * S:(r + 1) * S, c * S:(c + 1) * S] = pieces_np[i]
    return grid


def compute_ssim(
    asst: np.ndarray,
    perm: np.ndarray,
    ordered: torch.Tensor,     # [N, 3, S, S]
    rows: int, cols: int,
) -> float:
    """
    SSIM on grid images: ground-truth arrangement vs. model's arrangement.
    asst[i] = shuffled index placed at position i
    perm[j] = original position of shuffled piece j
    """
    ordered_np = ordered.permute(0, 2, 3, 1).cpu().numpy()   # [N, S, S, 3]
    pred_np    = ordered_np[perm[asst]]                        # reorder by prediction
    gt_grid   = pieces_to_grid(ordered_np, rows, cols)
    pred_grid = pieces_to_grid(pred_np,   rows, cols)
    return float(skimage_ssim(gt_grid, pred_grid, channel_axis=-1, data_range=1.0))


@torch.no_grad()
def evaluate(
    model:   "PuzzleSolver",
    test_ds: PuzzleDataset,
    cfg:     Config,
    max_samples: Optional[int] = None,
) -> dict:
    """
    Run evaluation on the test split and report ACC / SSIM / Kendall τ / PR / NP.
    """
    model.eval()
    loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                        num_workers=cfg.NUM_WORKERS,
                        pin_memory=torch.cuda.is_available())

    accs, ssims, taus, nps = [], [], [], []
    n_perfect = 0

    for i, (shuffled, P_target, ordered) in enumerate(tqdm(loader, desc="Evaluating")):
        if max_samples is not None and i >= max_samples:
            break

        shuffled_dev = shuffled.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, _ = model(shuffled_dev)

        # Hard assignment via Hungarian
        asst = hungarian_decode(logits)[0]           # [N]

        # Ground-truth:
        #   gt_asst[i]  = correct shuffled index for position i
        #   perm[j]     = original position of shuffled piece j
        gt_asst = P_target[0].argmax(dim=1).cpu().numpy()   # [N]
        perm    = P_target[0].argmax(dim=0).cpu().numpy()   # [N]

        orig_at_pos = perm[asst]   # original position placed at each grid slot

        acc  = compute_acc(asst, gt_asst)
        ssim = compute_ssim(asst, perm, ordered[0], cfg.GRID_ROWS, cfg.GRID_COLS)
        tau  = compute_kendall_tau(orig_at_pos)
        np_  = compute_neighbor_perf(orig_at_pos, cfg.GRID_ROWS, cfg.GRID_COLS)

        accs.append(acc); ssims.append(ssim); taus.append(tau); nps.append(np_)
        if acc == 1.0:
            n_perfect += 1

    n = len(accs)
    results = {
        "ACC (%)":         np.mean(accs)  * 100,
        "SSIM":            np.mean(ssims),
        "Kendall τ":       np.mean(taus),
        "PR (%)":          n_perfect / n * 100 if n else 0,
        "NP (%)":          np.mean(nps)  * 100,
        "n_samples":       n,
    }

    print(f"\n{'='*45}")
    print(f"  {'Metric':<16} {'Value':>10}")
    print(f"  {'─'*28}")
    for k, v in results.items():
        if k != "n_samples":
            print(f"  {k:<16} {v:>10.4f}")
    print(f"  {'─'*28}")
    print(f"  {'Samples':<16} {n:>10d}")
    print(f"{'='*45}")

    return results


In [ ]:
# ── Load model and evaluate ────────────────────────────────────────────────────
#
# If you have already trained and just want to evaluate a saved checkpoint:
#
# model = PuzzleSolver(cfg).to(device)
# ckpt = torch.load(str(Path(cfg.CHECKPOINT_DIR) / "global_model.pt"),
#                   map_location=device)
# model.load_state_dict(ckpt["model_state"])
# print(f"Loaded model from epoch {ckpt['epoch']}")
#
# train_ds, test_ds = make_datasets(cfg)
# results = evaluate(model, test_ds, cfg)


# ── Visualise a few test examples ─────────────────────────────────────────────

@torch.no_grad()
def visualise_examples(
    model:   "PuzzleSolver",
    test_ds: PuzzleDataset,
    cfg:     Config,
    n_show:  int = 4,
):
    """
    Display n_show side-by-side comparisons:
        Left  : ground-truth grid image
        Right : model reconstruction
    """
    model.eval()
    indices = random.sample(range(len(test_ds)), min(n_show, len(test_ds)))

    fig, axes = plt.subplots(n_show, 2, figsize=(10, n_show * 3))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for ax_row, idx in zip(axes, indices):
        shuffled, P_target, ordered = test_ds[idx]
        shuffled_dev = shuffled.unsqueeze(0).to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, _ = model(shuffled_dev)

        asst = hungarian_decode(logits)[0]
        perm = P_target.argmax(dim=0).cpu().numpy()
        acc  = compute_acc(asst, P_target.argmax(dim=1).cpu().numpy())

        ordered_np = ordered.permute(0, 2, 3, 1).cpu().numpy()  # [N, S, S, 3]
        pred_np    = ordered_np[perm[asst]]

        gt_grid   = pieces_to_grid(ordered_np, cfg.GRID_ROWS, cfg.GRID_COLS)
        pred_grid = pieces_to_grid(pred_np,   cfg.GRID_ROWS, cfg.GRID_COLS)

        ax_row[0].imshow(np.clip(gt_grid,   0, 1)); ax_row[0].set_title("Ground truth")
        ax_row[1].imshow(np.clip(pred_grid, 0, 1)); ax_row[1].set_title(f"Predicted  (ACC={acc*100:.1f}%)")
        for ax in ax_row:
            ax.axis("off")

    plt.suptitle("Puzzle reconstruction examples", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


# visualise_examples(model, test_ds, cfg, n_show=4)
